# Tutorial 10 - Data Fields And Field Types

This tutorial teaches how to think about BRAIN data fields before writing large Alpha batches.

It is offline-safe and uses example expressions only. It does not require access to live Data Explorer.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


## 1. The Data Explorer Checklist

Before using a field, inspect it in Data Explorer:

- field type
- coverage
- date coverage
- cadence
- region
- delay
- description
- units
- Alpha Count and User Count

The goal is to understand the field before choosing operators.


## 2. Matrix, Vector, And Group Fields

A Matrix data field has one value per instrument-date. A Vector data field has multiple values per instrument-date and should be reduced with operators such as `vec_avg`. A Group data field is a label used by group operators such as `group_rank`.

Sparse fields often need careful missing-data handling or operators such as `ts_backfill`.


In [ ]:
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record

excel_path = DATA_DIR / "tutorial_10_datafield_examples.xlsx"
alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
payload_records = []
for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    payload_records.append(
        record.__class__(
            row_id=record.row_id,
            alpha_hash=f"field-hash-{index}",
            payload=record.payload,
            metadata=record.metadata,
        )
    )

print(f"Loaded {len(payload_records)} payload records from {excel_path.name}")
[(record.row_id, record.alpha_hash, record.payload["settings"]["universe"]) for record in payload_records]


## 3. Classify The Example Expressions

This is a lightweight static exercise. The library does not validate live field metadata offline, so the notebook teaches the reasoning pattern instead.


In [ ]:
field_notes = {
    "field-001": {
        "field_type": "Matrix data field",
        "check": "coverage, cadence, region, delay",
        "operator_rule": "matrix fields can use rank and ts_mean directly",
    },
    "field-002": {
        "field_type": "Vector data field",
        "check": "confirm vector semantics in Data Explorer",
        "operator_rule": "reduce first with vec_avg, vec_sum, or vec_count",
    },
    "field-003": {
        "field_type": "Group data field",
        "check": "group size and peer meaning",
        "operator_rule": "use as the group argument in group_rank or group_neutralize",
    },
    "field-004": {
        "field_type": "Sparse matrix field",
        "check": "coverage, date coverage, and missingness",
        "operator_rule": "consider ts_backfill only when the data cadence supports it",
    },
}

for record in payload_records:
    note = field_notes[record.row_id]
    print(record.row_id, note["field_type"], "->", note["operator_rule"])


## 4. Payload Review

The API payload cannot tell you whether a field is truly matrix, vector, or group typed. That metadata comes from Data Explorer. The payload review still confirms that your expression, region, universe, delay, and settings are the ones you intended to simulate.


In [ ]:
for record in payload_records:
    print(record.row_id)
    print("  expression:", record.payload["regular"])
    print("  region:", record.payload["settings"]["region"])
    print("  delay:", record.payload["settings"]["delay"])
    print("  hash:", record.alpha_hash)


## 5. Beginner Field Rules

- Use Data Explorer before expression generation.
- For a Vector data field, reduce with `vec_avg`, `vec_sum`, `vec_count`, or another vector operator before normal ranking.
- For a Group data field, use it as a group label, not as a numeric signal.
- For sparse fields, inspect coverage before using `ts_backfill`.
- Match time-series windows to cadence.
- Prefer one field from a family first, then expand after a signal appears.
- Keep settings stable while testing field/operator changes.
